In [1]:
import pandas as pd

In [5]:
df_hearts = pd.read_csv('heart.csv')
df_datos_procesos = pd.read_csv('datos_procesos.csv',sep="|")

In [6]:
df_hearts.head(2)

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall,output
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1


In [7]:
df_datos_procesos.head(2)

,ID_Proceso,Uso_CPU,Uso_Memoria,Numero_Hilos,Tiempo_Ejecucion,Numero_Errores,Tipo_Proceso,Estado
0,1,37.454012,59.515562,16,8.184879,3,Aplicación,0
1,2,95.071431,36.471714,18,76.195256,8,Aplicación,0


In [10]:
# Información del DataFrame (tipos de datos, valores nulos)
df_datos_procesos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   ID_Proceso        1000000 non-null  int64  
 1   Uso_CPU           1000000 non-null  float64
 2   Uso_Memoria       1000000 non-null  float64
 3   Numero_Hilos      1000000 non-null  int64  
 4   Tiempo_Ejecucion  1000000 non-null  float64
 5   Numero_Errores    1000000 non-null  int64  
 6   Tipo_Proceso      1000000 non-null  object 
 7   Estado            1000000 non-null  int64  
dtypes: float64(3), int64(4), object(1)
memory usage: 61.0+ MB


In [ ]:
# Resumen estadístico de las columnas numéricas
print(df_datos_procesos.describe())

In [ ]:
# Ver la cantidad de valores nulos por columna
df_datos_procesos.isnull().sum()

ID_Proceso          0
Uso_CPU             0
Uso_Memoria         0
Numero_Hilos        0
Tiempo_Ejecucion    0
Numero_Errores      0
Tipo_Proceso        0
Estado              0
dtype: int64

In [13]:
# Distribución de las variables categóricas
df_datos_procesos["Tipo_Proceso"].value_counts()

Tipo_Proceso
Servicio      334085
Aplicación    332961
Sistema       332954
Name: count, dtype: int64

In [14]:
# Eliminar filas duplicadas
df_datos_procesos = df_datos_procesos.drop_duplicates()

In [15]:
# Eliminar filas con valores nulos
df_datos_procesos = df_datos_procesos.dropna()

In [18]:
from sklearn.preprocessing import OneHotEncoder

# Codificación One-Hot para la columna "Tipo_Proceso"
encoder = OneHotEncoder(sparse_output=False)  # Cambié 'sparse' por 'sparse_output'
tipo_proceso_encoded = encoder.fit_transform(df_datos_procesos[["Tipo_Proceso"]])

# Convertir la matriz codificada en un DataFrame
tipo_proceso_df = pd.DataFrame(tipo_proceso_encoded, columns=encoder.categories_[0])

# Unir las columnas codificadas con el DataFrame original
df_datos_procesos = df_datos_procesos.join(tipo_proceso_df)

# Eliminar la columna original "Tipo_Proceso"
df_datos_procesos = df_datos_procesos.drop(columns=["Tipo_Proceso"])



In [19]:
from sklearn.preprocessing import StandardScaler

# Columnas numéricas
numerical_cols = ["Uso_CPU", "Uso_Memoria", "Numero_Hilos", "Tiempo_Ejecucion", "Numero_Errores"]

# Inicializamos el escalador
scaler = StandardScaler()

# Escalar las características
df_datos_procesos[numerical_cols] = scaler.fit_transform(df_datos_procesos[numerical_cols])


In [20]:
from sklearn.model_selection import train_test_split

# Variable objetivo
X = df_datos_procesos.drop(columns=["Estado"])  # Características
y = df_datos_procesos["Estado"]  # Variable objetivo

# Dividir en conjunto de entrenamiento y prueba (80% - 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ver el tamaño de los conjuntos
print(f"Tamaño del conjunto de entrenamiento: {X_train.shape}")
print(f"Tamaño del conjunto de prueba: {X_test.shape}")


Tamaño del conjunto de entrenamiento: (800000, 9)
Tamaño del conjunto de prueba: (200000, 9)


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# Definir el modelo
lr = LogisticRegression()

# Entrenar el modelo
lr.fit(X_train, y_train)

# Predecir en el conjunto de prueba
y_pred = lr.predict(X_test)

# Evaluar el modelo
print(classification_report(y_test, y_pred))

# Calcular el AUC-ROC
roc_auc = roc_auc_score(y_test, y_pred)
print(f"Área bajo la curva ROC: {roc_auc}")


              precision    recall  f1-score   support

           0       0.97      0.99      0.98    187153
           1       0.73      0.55      0.63     12847

    accuracy                           0.96    200000
   macro avg       0.85      0.77      0.80    200000
weighted avg       0.95      0.96      0.96    200000

Área bajo la curva ROC: 0.7688868552999553


c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
from sklearn.tree import DecisionTreeClassifier

# Definir el modelo
dt = DecisionTreeClassifier(random_state=42)

# Entrenar el modelo
dt.fit(X_train, y_train)

# Predecir en el conjunto de prueba
y_pred_dt = dt.predict(X_test)

# Evaluar el modelo
print(classification_report(y_test, y_pred_dt))

# Calcular el AUC-ROC para el árbol de decisión
roc_auc_dt = roc_auc_score(y_test, y_pred_dt)
print(f"Área bajo la curva ROC (Árbol de Decisión): {roc_auc_dt}")


              precision    recall  f1-score   support

           0       0.97      0.96      0.97    187153
           1       0.51      0.53      0.52     12847

    accuracy                           0.94    200000
   macro avg       0.74      0.75      0.74    200000
weighted avg       0.94      0.94      0.94    200000

Área bajo la curva ROC (Árbol de Decisión): 0.7476620090601269


In [23]:
from sklearn.model_selection import cross_val_score

# Realizar validación cruzada (5 pliegues)
cross_val_scores = cross_val_score(lr, X, y, cv=5, scoring="roc_auc")

# Promediar los resultados de la validación cruzada
print(f"Puntuación media AUC-ROC de la validación cruzada: {cross_val_scores.mean()}")


c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or

Puntuación media AUC-ROC de la validación cruzada: 0.9703865416947091


In [24]:
# Importancia de las características del Árbol de Decisión
print(f"Importancia de las características: {dt.feature_importances_}")

# Coeficientes del modelo de regresión logística
print(f"Coeficientes del modelo de regresión logística: {lr.coef_}")


Importancia de las características: [0.09466645 0.24609569 0.27115944 0.06693964 0.17020609 0.04112912
 0.01783262 0.00298791 0.08898305]
Coeficientes del modelo de regresión logística: [[-1.79203582e-08  3.31779299e+00  2.10878414e+00  2.02963311e-02
   1.01389305e+00  1.17868194e-01 -1.97816966e+00 -3.41670352e+00
   5.63597039e-02]]
